<b>Galois Theory through Galois’ eyes.</b> Galois thought about ojects like field extensions, primitive elements, permutations, and the Galois group (indeed, groups in general) in a much different manner from the modern one that we use in this class. In this project, you’ll go back to Galois’ memoir and see how his ideas work from our modern view

In [ ]:
import numpy as onp
import sympy as sp
from pathlib import Path
import sys
_project_root: Path = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[1]))
import src.symmetry as sym
from src.categorization import show_latex_bmatrix, show_latex_eqn, latex_bmatrix_vec, cyclic_power_rep
from src.bases import CanonBasis4x4
from src.permutations import Permutation, PermutationGroup, PermElt
import src.permutations as perms
NDArray = onp.ndarray



class GaloisGroup(sym.FiniteGroupAction):
  pass

Gal = GaloisGroup


## Galois Resolvents
1. Galois resolvents and their translation to field theory (part B of section 12.2 in Cox). Summarize important defintiions, theorems, and proofs describing arguments in your own words as much as possible. Give at least one example not presented in the text.

### Intro

The point is to understand the splitting field of a separable polynomial. To do this, Galois used a variation of Lagrange's notation of resolvent polynomial. Suppose that 
$f\in F[x]$ can be written 
$f=a_0(x-\alpha_1)\cdots(x-\alpha_n)$ in a plitting field $L$, where $\alpha_1,\dots,\alpha_n$ are distinct. We also assume $F$ is infinte (why?)

THen, given $t_1,\dots, t_n\in F$, consider the polynomial of degree $n!$ defined by
$$
s(y)=\prod_{\sigma\in S_n} (y-(t_1\alpha_{\sigma(1)}+\cdots + t_n\alpha_{\sigma(n)}))
$$

Importantly, $s(y)\in F[y]$, which gives us......., and the proof of which uses symmetric functions and is similar to Galois's.

Galois asserts that since $\alpha_1,\dots,\alpha_n$ are distinct, one can find $t_1,\dots,t_n\in F$ so that the $n!$ elements
$$
t_1\alpha_{\sigma(1)}+\cdots + t_n\alpha_{\sigma(n)},\quad\sigma\in S_n
$$
are all distinct. In other words, we may choose suitable $t_1,\dots,t_n\in F$ such that $s(y)$ is separable. When this happens, we call $s(y)$ a Galois resolvent of $f$. 


### Symmetric Polynomials

A symmetric polynomial is what you might expect, one that is invariant under exchange of its indeterminates.

In [ ]:
# TODO can implement custom symbolic library using JAX operation tree?
n = 5
Sn = PermutationGroup.symmetric(n, name=f"S_{n}")
y = sp.Symbol("y")
roots = sp.symbols(fr"\alpha_0:{n}")

coeffs = onp.array([1,2,4,8,16])

def phi(*a):
  return onp.dot(coeffs, onp.array(a))
  # return sum(c * x for c, x in zip(coeffs, a, strict=True))



# indices = tuple(range(len(roots)))
# perms.perm_hom_tup(indices, image_fn=lambda i: i)
# Then, want Sn to act on the indices of the roots and to find the resolvent, take the product over all these
orbit_vals = []
seen = set()
for σ in Sn:
  val = phi(*perms.substitute_roots(σ, roots))
  if val not in seen:
    seen.add(val)
    orbit_vals.append(val)

R_factored = sp.prod(y - v for v in orbit_vals)
display(R_factored)

In [ ]:
R = sp.expand_mul(R_factored)
display(R)

**Example 12.2.1** We will compute a Galois resolvent of $f=(x^2-2)(x^2-3)=x^4-5x^2+6\in\mathbb{Q}[x]$, which has roots $\sqrt 2, -\sqrt 2, \sqrt 3,$ and $-\sqrt 3$. Let $(t_1,t_2,t_3,t_4)=(0,1,2,4)$.

In [22]:

coeffs_f = onp.array([1, 0, -5, 0, 6])

def f(*a):
  return onp.dot(coeffs_f, onp.array(a))

roots_f = onp.array([sp.sqrt(2), -sp.sqrt(2), sp.sqrt(3), -sp.sqrt(3)])
ts = onp.array([0,1,2,4])

def V(*a):
  return onp.dot(ts, onp.array(a))

# V = onp.dot(ts, roots_f)
display(V(*roots_f))

n = 4
Sn = PermutationGroup.symmetric(n, name=f"S_{n}")
y = sp.Symbol("y")

orbit_vals = []
seen = set()
for σ in Sn:
  val = V(*perms.substitute_roots(σ, roots_f))
  if val not in seen:
    seen.add(val)
    orbit_vals.append(val)
R_factored = sp.prod(y - v for v in orbit_vals)
display(R_factored)
R = sp.expand_mul(R_factored)
display(R)
display(sp.factor(R))


-2*sqrt(3) - sqrt(2)

(y - 4*sqrt(2) - sqrt(3))*(y - 4*sqrt(2) + sqrt(3))*(y - 3*sqrt(2) - 2*sqrt(3))*(y - 3*sqrt(2) + 2*sqrt(3))*(y - 2*sqrt(2) - sqrt(3))*(y - 2*sqrt(2) + sqrt(3))*(y - 2*sqrt(2) + 3*sqrt(3))*(y - sqrt(2) + 2*sqrt(3))*(y - sqrt(2) + 4*sqrt(3))*(y + sqrt(2) + 2*sqrt(3))*(y + sqrt(2) + 4*sqrt(3))*(y + 2*sqrt(2) + 3*sqrt(3))*(y - 4*sqrt(3) - sqrt(2))*(y - 4*sqrt(3) + sqrt(2))*(y - 3*sqrt(3) - 2*sqrt(2))*(y - 3*sqrt(3) + 2*sqrt(2))*(y - 2*sqrt(3) - sqrt(2))*(y - 2*sqrt(3) + sqrt(2))*(y - 2*sqrt(3) + 3*sqrt(2))*(y - sqrt(3) + 2*sqrt(2))*(y - sqrt(3) + 4*sqrt(2))*(y + sqrt(3) + 2*sqrt(2))*(y + sqrt(3) + 4*sqrt(2))*(y + 2*sqrt(3) + 3*sqrt(2))

y**24 - 350*y**22 + 52395*y**20 - 4390200*y**18 + 226512195*y**16 - 7470312150*y**14 + 158533048725*y**12 - 2128033120500*y**10 + 17319964832940*y**8 - 79514980673600*y**6 + 185487963684016*y**4 - 182187606350400*y**2 + 57817774440000

(y**4 - 100*y**2 + 2116)*(y**4 - 70*y**2 + 361)*(y**4 - 70*y**2 + 841)*(y**4 - 60*y**2 + 36)*(y**4 - 28*y**2 + 100)*(y**4 - 22*y**2 + 25)

Lagrange studies *rational functions* in formal roots $\varphi(x_1,\dots, x_n)$ and asks how many different values arise when the $x_i$ are permuted. 

Galois instead starts with a **specific polynomial**
$$
f\in F[x],
$$
its roots $\alpha_1,\dots,\alpha_n$ and its splitting field $L=F(\alpha_1,\dots,\alpha_n)$. Then, the relevant permutations are not all of $S_n$, but only those which perserve algebraic relations over $F$. 

Ex: For a cubic $f$, the discriminant detects whether the Galois group lies in $A_3$
$$
\Delta\in F^{\times 2} \iff \mathrm{Gal}(f/F)\subseteq A_3,
$$
so an irreducible cubic has Galois group
$$
S_3
$$
if the discriminant is not a square, and 
$$
A_3\cong C_3
$$
if the discriminant is a square.


For quartics, we use a cubic resolvent to distinguish possible groups such as 
$$
S_4,\,A_4,\,D_4,\,V_4,\,C_4
$$